What is CrewAI?

CrewAI is an agent-based orchestration framework designed to let multiple AI agents work together like a real-world team. Instead of a single large language model doing everything, CrewAI organizes roles, responsibilities, and workflows—similar to how humans collaborate in workplaces.

Think of it as building a team of AI specialists, where each agent has:

A role (what they are)

A goal (what they must accomplish)

A backstory (context that influences behavior)

A specific task (work assigned to them)

Optionally, tools and memory

# Agents to Research and Write an Article

Foundational concepts of multi-agent systems and get an overview of the crewAI framework.

For running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [3]:
# !pip install crewai==0.28.8 crewai_tools==0.1.6


In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [4]:
from crewai import Agent, Task, Crew

In [5]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

2026-10-25 15:56:59,297 - 8647401024 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 32
2026-10-25 15:56:59,299 - 8647401024 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 33
2026-10-25 15:56:59,300 - 8647401024 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 34


## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner


In [6]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
    llm=llm,
	verbose=True
)

### Agent: Writer

In [7]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

### Agent: Editor

In [8]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [9]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [10]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [11]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 1 or 2 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [12]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see here.

In [13]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer  
Final Answer: 

**Content Plan Document for Blog Article on Artificial Intelligence**

---

### 1. Latest Trends, Key Players, and Noteworthy News in Artificial Intelligence

**Trends:**
- **Generative AI:** The rise of AI models that can create text, images, and music, such as OpenAI's ChatGPT and DALL-E.
- **AI Ethics and Regulation:** Increasing focus on ethical AI practices and the development of regulations to govern AI use.
- **AI in Healthcare:** Advancements in AI applications for diagnostics, personalized medicine

- Display the results of your execution as markdown in the notebook.

In [14]:
from IPython.display import Markdown
Markdown(result)

```markdown
# Understanding Artificial Intelligence: Trends, Players, and Future Implications

Artificial Intelligence (AI) has rapidly evolved from a niche area of research to a transformative force across various sectors. As we navigate through 2023, understanding AI's significance and its implications on our daily lives is more crucial than ever. This article aims to shed light on the latest trends, key players, and future implications of AI, providing insights for tech enthusiasts, business professionals, healthcare providers, and the general public alike.

## What is Artificial Intelligence?

Artificial Intelligence refers to the simulation of human intelligence in machines programmed to think and learn like humans. It encompasses various technologies, including machine learning, natural language processing (NLP), and robotics. AI can be categorized into two main types: Narrow AI, which is designed for specific tasks (like voice assistants), and General AI, which possesses the ability to understand and reason across a wide range of tasks, similar to human intelligence.

The history of AI dates back to the mid-20th century, with significant milestones such as the development of the first neural networks and the introduction of machine learning algorithms. Over the decades, AI has transitioned from theoretical concepts to practical applications, leading to the sophisticated systems we see today. As AI continues to evolve, it is essential to stay informed about its advancements and implications.

## Current Trends in AI

### Generative AI and Its Applications

One of the most exciting trends in AI is the rise of generative AI, which refers to models capable of creating text, images, and even music. OpenAI's ChatGPT and DALL-E are prime examples of generative AI, showcasing the potential for creativity and innovation. These models have applications in content creation, design, and entertainment, revolutionizing how we interact with technology.

### The Role of AI in Healthcare and Diagnostics

AI's impact on healthcare is profound, with advancements in diagnostics, personalized medicine, and patient management. AI algorithms can analyze medical data to identify patterns and predict outcomes, leading to more accurate diagnoses and tailored treatment plans. As healthcare professionals increasingly adopt AI tools, the potential for improved patient care and operational efficiency becomes evident.

### Automation and Its Impact on Various Industries

The integration of AI in automation is transforming industries such as manufacturing and logistics. By enhancing productivity and efficiency, AI-driven automation allows businesses to streamline operations and reduce costs. However, this trend also raises concerns about job displacement, necessitating discussions around workforce adaptation and reskilling.

## Key Players in the AI Space

### Overview of Leading Companies and Their Contributions

Several key players are shaping the AI landscape. OpenAI is renowned for its cutting-edge generative models, while Google DeepMind focuses on AI research across various fields. IBM Watson is pioneering AI solutions for business and healthcare, and Microsoft is heavily investing in AI tools and cloud services. NVIDIA leads in AI hardware and software solutions, providing the infrastructure necessary for AI advancements.

### Emerging Startups to Watch

In addition to established companies, numerous startups are making waves in the AI sector. These emerging players often bring innovative solutions and fresh perspectives, contributing to the dynamic nature of the AI landscape. Keeping an eye on these startups can provide valuable insights into the future of AI technology.

## Ethics and Regulation in AI

### Discussion on the Importance of Ethical AI

As AI technologies become more pervasive, the importance of ethical AI practices cannot be overstated. Concerns about bias, privacy, and accountability are at the forefront of discussions surrounding AI development. Ensuring that AI systems are designed and implemented ethically is crucial for building trust and fostering responsible innovation.

### Overview of Current Regulations and Proposed Laws

Governments worldwide are beginning to recognize the need for regulations governing AI use. Recent legislative developments aim to establish frameworks that promote ethical AI practices while addressing potential risks. Staying informed about these regulations is essential for businesses and individuals alike, as they will shape the future landscape of AI.

## Future of AI

### Predictions for AI Advancements in the Next 5-10 Years

Looking ahead, the future of AI is promising. Experts predict significant advancements in AI capabilities, including improved natural language understanding, enhanced decision-making processes, and more sophisticated generative models. These developments will likely lead to increased integration of AI in everyday life, transforming how we work, communicate, and interact with technology.

### Potential Societal Impacts and Changes

The societal implications of AI advancements are profound. While AI has the potential to drive economic growth and improve quality of life, it also raises questions about job displacement and ethical considerations. Engaging in discussions about the future of work and the role of AI in society is essential for navigating these changes responsibly.

## Conclusion

Understanding Artificial Intelligence is vital in today's rapidly evolving technological landscape. By staying informed about the latest trends, key players, and ethical considerations, we can better navigate the complexities of AI and its implications for our lives. As we move forward, it is crucial to engage with these developments and advocate for responsible AI practices.

### Call to Action

We invite you to subscribe for updates on AI trends and developments. Share this article on social media to help spread awareness about the importance of understanding AI and its impact on our future.
```

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

In [ ]:
Markdown(result)